# Rețele de calculatoare · Săptămâna 1 — Notebook de aplicații (Google Colab)
### Introducere: ce este o rețea și cum este alcătuit Internetul

Notebook însoțitor al **pachetului didactic din Săptămâna 1** (Cursul 1, Seminarul 1, Laboratorul 1 Linux, Laboratorul 1 Windows).
Fiecare parte a pachetului are aici o parte corespunzătoare, pe care o puteți rula:

| Partea | În pachetul didactic | În acest notebook |
|---|---|---|
| **A** | Cursul 1 | Calculatoare și simulări pentru fiecare formulă și exemplu rezolvat; exercițiile de la sfârșitul cursului verificate prin cod |
| **B** | Seminarul 1: raționament cu numere | Problemele 1–8 rezolvate în Python, plus un simulator de coadă, un *protocol de oră* funcțional (client + server) și un analizor de traceroute |
| **C** | Laboratorul 1 (Linux) | Sarcinile 1–8 rulate pe mașina virtuală Colab, care este o gazdă Linux reală dintr-un centru de date Google |
| **D** | Laboratorul 1 (Windows) | Aceleași sarcini, în modul de gândire Windows/PowerShell, cu Python (și, opțional, PowerShell 7 real) |

**Cum se folosește.** Rulați celulele în ordine (merge și `Runtime → Run all`). Celulele marcate cu ✍️ sunt pentru răspunsurile voastre la întrebările Q1–Q16, astfel încât notebook-ul să poată fi predat ca raport (`File → Download → .ipynb`).

> **Despre mașina Colab.** Colab vă oferă o mașină virtuală Linux într-un centru de date Google, nu un calculator din laboratorul facultății. Adresele, gateway-ul și rutele vor fi cele ale acelui centru de date. Unele rețele cloud blochează ICMP (pachetele folosite de `ping` și de unele probe `traceroute`). Acolo unde se întâmplă asta, notebook-ul trece la măsurători bazate pe TCP și vă spune de ce.

## 0 · Pregătire

In [ ]:
# ---------------------------------------------------------------
#  Funcții ajutătoare folosite în tot notebook-ul. Rulați întâi această celulă.
# ---------------------------------------------------------------
import os, re, sys, time, math, json, socket, shutil, subprocess, threading, ipaddress
from collections import deque, Counter

IN_COLAB = 'google.colab' in sys.modules
SUDO = '' if os.geteuid() == 0 else 'sudo '

def sh(cmd, timeout=60, show=True):
    """Rulează o comandă shell, afișează ieșirea (stdout+stderr), nu ridică niciodată excepții."""
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        out = (r.stdout or '') + (r.stderr or '')
    except subprocess.TimeoutExpired as e:
        o = e.stdout or ''
        out = (o.decode(errors='replace') if isinstance(o, bytes) else o) + f'\n[oprit după {timeout} s]'
    if show:
        print(f'$ {cmd}\n{out.rstrip()}\n')
    return out

def have(tool):
    return shutil.which(tool) is not None

def default_route():
    """(interfața, gateway-ul) folosite pentru a ajunge în Internet."""
    out = sh('ip route get 1.1.1.1', show=False)
    dev = re.search(r'\bdev (\S+)', out)
    via = re.search(r'\bvia (\S+)', out)
    src = re.search(r'\bsrc (\S+)', out)
    return (dev.group(1) if dev else 'any', via.group(1) if via else None, src.group(1) if src else None)

def resolve(host):
    try:
        return socket.gethostbyname(host)
    except Exception as e:
        print(f'nu pot rezolva {host}: {e}')
        return None

def tcp_rtt(host, port=443, n=5, timeout=3):
    """Măsoară RTT ca durată a unui connect() TCP (un schimb SYN / SYN-ACK).
    Funcționează și acolo unde ping-ul ICMP este blocat (ca pe multe VM-uri din cloud)."""
    ip = resolve(host)
    if not ip:
        return []
    rtts = []
    for _ in range(n):
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(timeout)
        t0 = time.perf_counter()
        try:
            s.connect((ip, port))
            rtts.append((time.perf_counter() - t0) * 1000)
        except Exception as e:
            print(f'  conectarea la {ip}:{port} a eșuat: {e}')
        finally:
            s.close()
        time.sleep(0.2)
    if rtts:
        print(f'RTT prin conectare TCP la {host} ({ip}:{port}): min/med/max = '
              f'{min(rtts):.2f}/{sum(rtts)/len(rtts):.2f}/{max(rtts):.2f} ms  ({len(rtts)}/{n} răspunsuri)')
    return rtts

def fmt_time(seconds):
    """Afișează frumos o durată."""
    s = abs(seconds)
    if s < 1e-6: return f'{seconds*1e9:.3g} ns'
    if s < 1e-3: return f'{seconds*1e6:.4g} µs'
    if s < 1:    return f'{seconds*1e3:.4g} ms'
    if s < 120:  return f'{seconds:.4g} s'
    if s < 7200: return f'{seconds/60:.4g} min'
    if s < 2*86400: return f'{seconds/3600:.4g} h'
    return f'{seconds/86400:.4g} zile'

def check(name, got, expected, rel=0.02):
    ok = math.isclose(got, expected, rel_tol=rel, abs_tol=1e-12)
    print(f"{'✅' if ok else '❌'} {name}: calculat {got:.6g}, așteptat ≈ {expected:.6g}")
    return ok

print('Funcțiile ajutătoare au fost încărcate. Rulează în Colab:', IN_COLAB, '| root:', os.geteuid() == 0)

In [ ]:
# Instalează uneltele de linie de comandă folosite în Partea C (circa 30 s în Colab).
pkgs = 'iproute2 iputils-ping traceroute mtr-tiny dnsutils curl netcat-openbsd tcpdump'
if os.geteuid() == 0 or have('sudo'):
    sh(f'{SUDO}apt-get -qq update > /dev/null 2>&1; DEBIAN_FRONTEND=noninteractive {SUDO}apt-get -qq install -y {pkgs} > /dev/null 2>&1; echo done', timeout=600)
for t in ['ip', 'ping', 'traceroute', 'mtr', 'dig', 'curl', 'nc', 'ss', 'tcpdump']:
    print(f'{t:11s}', '✅' if have(t) else '❌ lipsește')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 4.5), 'axes.grid': True, 'grid.alpha': .3})

---
# Partea A · Cursul 1 — cursul în cod

## A.1 Unități: convertiți întâi totul în biți
Ratele se exprimă în **biți pe secundă**, cu prefixe **zecimale** (1 Mbit/s = 10⁶ bit/s); dimensiunile fișierelor se exprimă în **octeți** (1 MB = 10⁶ B, dacă nu se spune altfel). Semnalele în cupru și fibră se propagă cu aproximativ **2 × 10⁸ m/s = 200 km/ms**.

In [ ]:
K, M, G, T = 1e3, 1e6, 1e9, 1e12
BYTE = 8                    # bits
S_FIBRE = 2e8               # m/s, cupru și fibră
S_AIR = 3e8                 # m/s, radio și vid

def bits(nbytes):           # octeți -> biți
    return nbytes * BYTE

def t_trans(L_bits, R_bps):  # întârzierea de transmisie L/R
    return L_bits / R_bps

def t_prop(d_m, s=S_FIBRE):  # întârzierea de propagare d/s
    return d_m / s

print('pachet de 1500 de octeți =', bits(1500), 'biți')
print('1 MiB / 1 MB =', 2**20 / 1e6, '→ unitățile binare sunt cu ~4,9 % mai mari')

## A.2 Comutarea de pachete: store-and-forward
Un comutator trebuie să primească întregul pachet înainte de a-l retransmite. Un pachet pe **N** legături de rată **R**: `d = N·L/R`. **P** pachete trimise unul după altul: `d = (N + P − 1)·L/R`, deoarece legăturile lucrează în paralel la pachete diferite (o *bandă de asamblare*).

In [ ]:
def store_and_forward(L_bits, R_bps, N_links, P_packets=1):
    return (N_links + P_packets - 1) * L_bits / R_bps

# Exemplul rezolvat 1 — pachet de 1500 de octeți, legături de 100 Mbit/s
L = bits(1500)
print('L/R pe o legătură        :', fmt_time(t_trans(L, 100*M)))
print('două legături, un router:', fmt_time(store_and_forward(L, 100*M, 2)))
check('Exemplul rezolvat 1', store_and_forward(L, 100*M, 2), 0.24e-3)

In [ ]:
def pipeline_chart(L_bits, R_bps, N_links, P_packets, title=''):
    """Diagramă Gantt: când ocupă fiecare pachet fiecare legătură (store-and-forward, fără propagare)."""
    tt = L_bits / R_bps
    fig, ax = plt.subplots(figsize=(9, 0.6 * N_links + 1.5))
    colors = plt.cm.tab10.colors
    for p in range(P_packets):
        for n in range(N_links):
            start = (p + n) * tt
            ax.barh(n, tt * 1e3, left=start * 1e3, color=colors[p % 10], edgecolor='k')
            ax.text((start + tt / 2) * 1e3, n, f'P{p+1}', ha='center', va='center', fontsize=8, color='w')
    ax.set_yticks(range(N_links), [f'legătura {n+1}' for n in range(N_links)])
    ax.invert_yaxis(); ax.set_xlabel('timp (ms)')
    total = store_and_forward(L_bits, R_bps, N_links, P_packets)
    ax.set_title(title or f'{P_packets} pachete pe {N_links} legături: ultimul bit sosește la {total*1e3:.3g} ms')
    plt.tight_layout(); plt.show()

# Exercițiul 1 din curs: cinci pachete de 1500 de octeți, trei legături de 10 Mbit/s
pipeline_chart(bits(1500), 10*M, 3, 5)

## A.3 Comutarea de circuite (TDM)
La multiplexarea în timp, fiecare circuit primește câte un interval din fiecare cadru — `R/intervale` — indiferent dacă celelalte intervale sunt folosite sau nu.

In [ ]:
def tdm_transfer(file_bits, R_bps, slots, setup_s=0.0):
    per_circuit = R_bps / slots
    return per_circuit, setup_s + file_bits / per_circuit

rate, total = tdm_transfer(640_000, 2.048*M, 32, setup_s=0.5)
print(f'Fiecare circuit primește {rate/1e3:.0f} kbit/s; fișierul durează {total:.1f} s (inclusiv 0,5 s de stabilire)')
check('Exemplul rezolvat 2', total, 10.5)

## A.4 Comutarea de pachete vs. comutarea de circuite: multiplexarea statistică
O legătură de 1 Gbit/s; fiecare utilizator trimite cu 100 Mbit/s când este activ și este activ 10 % din timp.
Comutarea de circuite admite 10 utilizatori. La comutarea de pachete, cererea depășește capacitatea doar când **11 sau mai mulți** utilizatori sunt activi simultan. Numărul utilizatorilor activi urmează o distribuție binomială:

$$P(X \ge 11) = \sum_{k=11}^{N} \binom{N}{k}\,p^k (1-p)^{N-k}$$

In [ ]:
def p_overload(n_users, p_active, max_active):
    """P(mai mult de max_active utilizatori activi simultan) — coada binomială."""
    return sum(math.comb(n_users, k) * p_active**k * (1 - p_active)**(n_users - k)
               for k in range(max_active + 1, n_users + 1))

link, user_rate, p = 1*G, 100*M, 0.10
cap = int(link // user_rate)                    # utilizatori pe care legătura îi poate purta simultan
print('Comutarea de circuite suportă', cap, 'utilizatori')
pr = p_overload(35, p, cap)
print(f'Comutarea de pachete, 35 de utilizatori: P(11 sau mai mulți activi) = {pr:.5f}')
check('Exemplul rezolvat 3', pr, 0.0004, rel=0.15)

# Monte Carlo: simulăm un milion de momente
rng = np.random.default_rng(1)
active = rng.binomial(35, p, size=1_000_000)
print(f'Simulare: supraîncărcare în {np.mean(active > cap):.5f} dintr-un milion de momente aleatoare')

In [ ]:
Ns = np.arange(10, 81)
probs = [p_overload(n, p, cap) for n in Ns]
plt.semilogy(Ns, probs, 'o-', ms=3)
plt.axvline(10, color='r', ls='--', label='limita comutării de circuite (10 utilizatori)')
plt.axhline(1e-3, color='gray', ls=':', label='0,1 % din timp')
plt.xlabel('numărul de utilizatori care împart legătura de 1 Gbit/s'); plt.ylabel('P(cerere > capacitate)')
plt.title('Multiplexare statistică: câți utilizatori pot împărți o legătură?'); plt.legend(); plt.show()
best = max(n for n, q in zip(Ns, probs) if q < 1e-3)
print(f'Până la {best} utilizatori țin supraîncărcarea sub 0,1 % — de {best/cap:.1f}× mai mulți decât admite comutarea de circuite.')

In [ ]:
# Exercițiul 3 din curs: toți cei 35 de utilizatori activi simultan
print(f'P(toți 35 activi) = 0.1^35 = {0.1**35:.1e}  — practic niciodată.')
print('Riscul relevant este „puțin mai mulți utilizatori decât poate purta legătura”, adică 0,0004 de mai sus.')

## A.5 Cele patru întârzieri și debitul
`d_nodal = d_proc + d_coadă + d_trans + d_prop`, cu `d_trans = L/R` (depinde de lungimea pachetului și de rata legăturii, **nu** de distanță) și `d_prop = d/s` (depinde de distanță, **nu** de lungimea pachetului sau de rată).

In [ ]:
def nodal_delay(L_bits, R_bps, d_m, d_proc=0.0, d_queue=0.0, s=S_FIBRE):
    parts = {'procesare': d_proc, 'coadă': d_queue,
             'transmisie': t_trans(L_bits, R_bps), 'propagare': t_prop(d_m, s)}
    for k, v in parts.items():
        print(f'  {k:13s} {fmt_time(v)}')
    total = sum(parts.values())
    print(f'  {"TOTAL":13s} {fmt_time(total)}')
    return parts

print('Exemplul rezolvat 4 — Cluj-Napoca → București, 450 km de fibră, 100 Mbit/s, pachet de 1500 de octeți')
p4 = nodal_delay(bits(1500), 100*M, 450e3)
print(f'Propagare / transmisie = {p4["propagare"]/p4["transmisie"]:.1f}')
print('\nAcelași pachet pe o legătură de 1 Mbit/s:')
nodal_delay(bits(1500), 1*M, 450e3);

In [ ]:
# Care domină? Transmisie vs. propagare când se schimbă rata legăturii (cale de 450 km)
rates = np.logspace(5, 10, 200)         # 100 kbit/s ... 10 Gbit/s
plt.loglog(rates, bits(1500) / rates * 1e3, label='transmisie L/R (1500 B)')
plt.loglog(rates, np.full_like(rates, t_prop(450e3) * 1e3), label='propagare d/s (450 km)')
plt.xlabel('rata legăturii (bit/s)'); plt.ylabel('întârziere (ms)'); plt.legend()
plt.title('Cu cât legătura e mai rapidă, cu atât distanța domină mai mult'); plt.show()

In [ ]:
# Analogia convoiului din curs
cars, booth_s, road_km, speed_kmh = 10, 12, 100, 100
print(f'Transmisie (toate mașinile prin ghișeu): {cars*booth_s} s = 2 min')
print(f'Propagare (condus {road_km} km cu {speed_kmh} km/h): {road_km/speed_kmh:.0f} h')

# Debitul: decide legătura-gâtuire
def transfer_time(file_bytes, link_rates_bps):
    bottleneck = min(link_rates_bps)
    return bits(file_bytes) / bottleneck, bottleneck

t, bn = transfer_time(8e6, [100*M, 2*M, 1*G])
print(f'\nFișier de 8 MB, gâtuire {bn/M:.0f} Mbit/s → cel puțin {t:.0f} s')

## A.6 Stratificarea și încapsularea într-o singură imagine
Fiecare strat adaugă antetul său în fața a ceea ce primește de la stratul de deasupra; receptorul le îndepărtează în ordine inversă. Gazdele implementează 5 straturi, routerele 3, comutatoarele de nivel legătură 2.

In [ ]:
STACK = [  # stratul, numele PDU, antetul adăugat (dimensiune ilustrativă în octeți)
    ('Aplicație',   'mesaj',    None),
    ('Transport',   'segment',  ('H_t', 20)),   # de ex. TCP
    ('Rețea',       'datagramă', ('H_n', 20)),  # de ex. IPv4
    ('Legătură',    'cadru',    ('H_l', 18)),   # de ex. antet + trailer Ethernet
    ('Fizic',       'biți',     None),
]

def encapsulate(message):
    unit, size = [f'[{message}]'], len(message)
    print(f'{"Aplicație":12s} mesaj    : {"".join(unit)}   ({size} B)')
    for layer, pdu, hdr in STACK[1:4]:
        unit.insert(0, f'[{hdr[0]}]'); size += hdr[1]
        print(f'{layer:12s} {pdu:9s}: {"".join(unit)}   ({size} B)')
    print(f'{"Fizic":12s} {size*8} biți pe fir')
    return size

total = encapsulate('GET /index.html')
DEVICES = {'gazdă': 5, 'router': 3, 'comutator de nivel legătură': 2}
print()
for dev, n in DEVICES.items():
    print(f'{dev.capitalize():28s} implementează straturile 1–{n}: {", ".join(l for l, _, _ in STACK[::-1][:n])}')

## A.7 O rețea de rețele
Un model de jucărie al Figurii 1: ISP-urile de nivel 1 (tier-1) fac peering între ele, ISP-urile regionale cumpără tranzit de la ele, ISP-urile de acces cumpără tranzit de la cele regionale, unele rețele fac și **peering** la un IXP, iar un furnizor de conținut se conectează direct la ISP-urile de acces. Numărăm câte rețele administrate independent traversează un pachet.

In [ ]:
import networkx as nx
G_ = nx.Graph()
customer_provider = [('Access-A', 'Regional-1'), ('Access-B', 'Regional-1'), ('Access-C', 'Regional-2'),
                     ('Access-D', 'Regional-2'), ('Regional-1', 'Tier1-X'), ('Regional-2', 'Tier1-Y'),
                     ('Regional-2', 'Tier1-X'),                     # multi-homing
                     ('Content-CDN', 'Tier1-Y')]
peering = [('Tier1-X', 'Tier1-Y'), ('Access-B', 'Access-C'),        # peering la un IXP
           ('Content-CDN', 'Access-A'), ('Content-CDN', 'Access-D')]
G_.add_edges_from(customer_provider, kind='transit'); G_.add_edges_from(peering, kind='peering')

pos = {'Tier1-X': (-1, 3), 'Tier1-Y': (1, 3), 'Regional-1': (-1.5, 2), 'Regional-2': (1.5, 2),
       'Access-A': (-2.5, 1), 'Access-B': (-0.8, 1), 'Access-C': (0.8, 1), 'Access-D': (2.5, 1), 'Content-CDN': (0, 0)}
plt.figure(figsize=(9, 5))
nx.draw_networkx_nodes(G_, pos, node_color='#cfe3ff', node_size=1800, edgecolors='k')
nx.draw_networkx_labels(G_, pos, font_size=8)
nx.draw_networkx_edges(G_, pos, edgelist=customer_provider, width=2)
nx.draw_networkx_edges(G_, pos, edgelist=peering, style='dashed', width=2, edge_color='tab:orange')
plt.title('Linie continuă: legături plătite client–furnizor · întreruptă: peering'); plt.axis('off'); plt.show()

for a, z in [('Access-A', 'Access-D'), ('Access-B', 'Access-C'), ('Access-A', 'Content-CDN')]:
    path = nx.shortest_path(G_, a, z)
    print(f'{a} → {z}: {" → ".join(path)}   ({len(path)} rețele)')

## A.8 Exercițiile de la sfârșitul cursului, verificate prin cod
Încercați-le întâi pe hârtie, apoi rulați celula.

In [ ]:
print('Exercițiul 1 — cinci pachete de 1500 de octeți, trei legături de 10 Mbit/s')
t1 = store_and_forward(bits(1500), 10*M, 3, 5)
check('ultimul bit al ultimului pachet (s)', t1, 8.4e-3)

print('\nExercițiul 2 — satelit geostaționar, la 36 000 km altitudine')
up = 36_000e3 / S_AIR
check('sol → satelit (s)', up, 0.12); check('urcare și coborâre (s)', 2*up, 0.24)
print('  O întrebare și răspunsul ei sunt la ~0,5 s distanță: incomod pentru o convorbire telefonică.')

print('\nExercițiul 3 — toți cei 35 de utilizatori activi')
print(f'  0.1^35 = {0.1**35:.0e}')

print('\nExercițiul 4 — 40 TB de la Iași la București: legătură de 100 Mbit/s vs. curier (8 h)')
t_link = bits(40*T) / (100*M)
print(f'  legătura: {fmt_time(t_link)}   curierul: 8 h  →  câștigă curierul')
check('timp pe legătură (zile)', t_link/86400, 37, rel=0.02)

print('\nExercițiul 5 — cât de lung este un pachet de 12 000 de biți pe fibră de 100 Mbit/s?')
length_m = t_trans(12_000, 100*M) * S_FIBRE
check('lungimea pachetului (km)', length_m/1e3, 24)

**Întrebări de recapitulare** (din notele de curs — răspundeți cu propriile cuvinte)
1. Gazdă vs. comutator de pachete; este un router Wi-Fi de acasă o gazdă? · 2. Cele trei lucruri pe care le definește un protocol, în exemplul HTTP · 3. De ce sunt legăturile de acces asimetrice? · 4. De ce nu poate un router să retransmită o jumătate de pachet? · 5. O aplicație servită mai bine de comutarea de circuite · 6. Retransmitere (forwarding) vs. rutare · 7. De ce fac ISP-urile peering gratuit? · 8. Ce întârziere se schimbă la trecerea de la 100 Mbit/s la 1 Gbit/s? · 9. Ce straturi implementează un router?

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **R1:** 
- **R2:** 
- **R3:** 
- **R4:** 
- **R5:** 
- **R6:** 
- **R7:** 
- **R8:** 
- **R9:** 

---
# Partea B · Seminarul 1 — Raționament despre rețele cu numere
Fiecare problemă este enunțată pe scurt; codul calculează fiecare răspuns și îl verifică față de soluția seminarului. Schimbați numerele pentru a explora.

## B.A Încălzire: unități

In [ ]:
print('(a) pachet de 1500 de octeți   :', bits(1500), 'biți')
print('(b) 5 MB la 20 Mbit/s          :', bits(5*M) / (20*M), 's')
print('(c) distanța în fibră în 1 ms  :', S_FIBRE * 1e-3 / 1e3, 'km')
print('(d) „4 MB/s” pe un telefon     :', 4 * 8, 'Mbit/s')

## B.1 Problema 1 — o singură legătură
A ↔ B: R = 10 Mbit/s, 2000 km, s = 2 × 10⁸ m/s, un pachet de 1500 de octeți.

In [ ]:
R, d, L = 10*M, 2000e3, bits(1500)
tt, tp = t_trans(L, R), t_prop(d)
check('1. întârzierea de transmisie (s)', tt, 1.2e-3); check('1. întârzierea de propagare (s)', tp, 10e-3)
check('2. ultimul bit sosește la (s)', tt + tp, 11.2e-3)
check('3. distanța primului bit față de A când A termină (km)', tt * S_FIBRE / 1e3, 240)
check('4. lungimea la care d_trans = d_prop (km)', S_FIBRE * L / R / 1e3, 240)
check('5. lungimea unui bit (m)', S_FIBRE / R, 20)

## B.2 Problema 2 — o cale de trei legături
Pachet de 12 000 de biți, A → R1 → R2 → B, 20 µs de procesare în fiecare router, cozi goale.

| Legătura | Rata | Lungimea |
|---|---|---|
| A – R1 (LAN-ul campusului) | 100 Mbit/s | 1 km |
| R1 – R2 (regională) | 10 Mbit/s | 500 km |
| R2 – B (centru de date) | 1 Gbit/s | 50 km |

In [ ]:
links = [('A–R1', 100*M, 1e3), ('R1–R2', 10*M, 500e3), ('R2–B', 1*G, 50e3)]
L, d_proc, n_routers = 12_000, 20e-6, 2
print(f'{"leg.":6s} {"trans":>10s} {"prop":>10s}')
for name, R_, d_ in links:
    print(f'{name:6s} {fmt_time(t_trans(L, R_)):>10s} {fmt_time(t_prop(d_)):>10s}')
trans = sum(t_trans(L, R_) for _, R_, _ in links)
prop = sum(t_prop(d_) for _, _, d_ in links)
total = trans + prop + n_routers * d_proc
print(f'transmisie {fmt_time(trans)} + propagare {fmt_time(prop)} + procesare {fmt_time(n_routers*d_proc)}')
check('1. întârzierea capăt la capăt (s)', total, 4.127e-3, rel=0.005)

bottleneck = min(R_ for _, R_, _ in links)
check('3. debitul maxim (Mbit/s)', bottleneck / M, 10)
check('3. un fișier de 30 MB durează cel puțin (s)', bits(30*M) / bottleneck, 24)
links_up = [('A–R1', 1*G, 1e3)] + links[1:]
print('4. după modernizarea A–R1 la 1 Gbit/s, gâtuirea rămâne',
      min(R_ for _, R_, _ in links_up) / M, 'Mbit/s → nicio accelerare')

## B.3 Problema 3 — de ce pachete? (segmentarea și prețul ei)
Mesaj de 1,2 × 10⁶ biți, trei legături de 2 Mbit/s, două comutatoare store-and-forward.

In [ ]:
Msg, R, N = 1.2e6, 2*M, 3
t_msg = store_and_forward(Msg, R, N, 1)
t_pkt = store_and_forward(Msg/100, R, N, 100)
t_hdr = store_and_forward(Msg/100 + 40*8, R, N, 100)
check('1. comutare de mesaje (s)', t_msg, 1.8)
check('2. 100 de pachete (s)', t_pkt, 0.612)
check('3. cu antete de 40 de octeți (s)', t_hdr, 0.628, rel=0.005)
print(f'   supraîncărcarea antetelor = {320/12320:.1%}')

# Extensie: ce dimensiune a pachetului minimizează timpul total?
P = np.arange(1, 2001)
times = (N + P - 1) * (Msg / P + 320) / R
best = P[np.argmin(times)]
plt.plot(P, times, label='cu antet de 40 de octeți per pachet')
plt.plot(P, (N + P - 1) * (Msg / P) / R, '--', label='fără antet')
plt.axvline(best, color='r', ls=':', label=f'optim ≈ {best} pachete (câte {Msg/best/8:.0f} B de date)')
plt.xscale('log'); plt.xlabel('numărul de pachete P'); plt.ylabel('timp total (s)'); plt.legend()
plt.title('Prea puține pachete: fără paralelism · prea multe: prea multe antete'); plt.show()

## B.4 Problema 4 — circuite sau pachete?
Legătură de 100 Mbit/s, utilizatorii trimit cu 10 Mbit/s când sunt activi și sunt activi 20 % din timp.

In [ ]:
cap4 = int(100 // 10)
check('1. comutarea de circuite suportă (utilizatori)', cap4, 10)
check('2. utilizatori activi în medie (din 30)', 30 * 0.2, 6)
print('   încărcarea medie = 60 Mbit/s = 60 % din capacitate')
pr4 = p_overload(30, 0.2, cap4)
check('3. P(mai mult de 10 din 30 activi)', pr4, 0.026, rel=0.05)

print('\nTemă: utilizatori activi 50 % din timp, 15 utilizatori')
pr_hw = p_overload(15, 0.5, cap4)
print(f'   P(mai mult de 10 din 15 activi) = {pr_hw:.3f} → supraîncărcare {pr_hw:.1%} din timp')

for pa, label in [(0.1, '10 % activi'), (0.2, '20 % activi'), (0.5, '50 % activi')]:
    ns = np.arange(10, 61)
    plt.semilogy(ns, [max(p_overload(n, pa, cap4), 1e-12) for n in ns], label=label)
plt.axhline(0.01, color='gray', ls=':'); plt.ylim(1e-8, 1.5)
plt.xlabel('utilizatori'); plt.ylabel('P(supraîncărcare)'); plt.legend()
plt.title('Cu cât traficul e mai în rafale, cu atât multiplexarea statistică câștigă mai mult'); plt.show()

## B.5 Problema 5 — se formează o coadă
Legătura de ieșire are 1 Mbit/s, pachetele au 12 000 de biți → `L/R = 12 ms`. Un mic simulator al unei cozi FIFO cu buffer finit reproduce toate cele trei cazuri.

In [ ]:
def fifo(arrivals, L_bits, R_bps, buffer_pkts=None):
    """Simulează o coadă FIFO de ieșire. Întoarce înregistrări per pachet și momentele aruncărilor.
    buffer_pkts = pachetele care pot AȘTEPTA (cel aflat în transmisie nu se numără)."""
    tx = L_bits / R_bps
    in_system = deque()              # momentele de plecare ale pachetelor încă aflate în router
    records, drops, last_dep = [], [], 0.0
    for a in arrivals:
        while in_system and in_system[0] <= a:
            in_system.popleft()
        waiting = max(len(in_system) - 1, 0)
        if buffer_pkts is not None and in_system and waiting >= buffer_pkts:
            drops.append(a); continue
        start = max(a, last_dep)
        last_dep = start + tx
        in_system.append(last_dep)
        records.append({'arrival': a, 'wait': start - a, 'queue_len': len(in_system) - 1})
    return records, drops

L, R = 12_000, 1*M
r1, _ = fifo([0]*5, L, R)
print('1. cinci pachete deodată — așteptări (ms):', [round(r['wait']*1e3) for r in r1],
      '| medie', np.mean([r['wait'] for r in r1])*1e3, 'ms')
r2, _ = fifo(np.arange(200) * 0.012, L, R)
print('2. unul la 12 ms — așteptarea maximă:', round(max(r['wait'] for r in r2)*1e3, 6), 'ms')
arr3 = np.arange(1000) * 0.010
r3, drops = fifo(arr3, L, R, buffer_pkts=20)
print(f'3. unul la 10 ms, buffer 20 — prima aruncare la t = {drops[0]:.2f} s; '
      f'apoi sunt aruncate {len(drops)/len(arr3[arr3 >= drops[0]]):.0%} din sosiri')
for case, gap in [('2', 0.012), ('3', 0.010)]:
    print(f'4. intensitatea traficului în cazul {case}: (L/interval)/R = {L/gap/R:.2f}')

t = [r['arrival'] for r in r3]; q = [r['queue_len'] for r in r3]
plt.step(t, q, where='post'); plt.axhline(20, color='r', ls='--', label='buffer = 20 de pachete')
plt.xlabel('timp (s)'); plt.ylabel('pachete în așteptare'); plt.legend()
plt.title('Intensitate 1,2: coada crește până se umple bufferul, apoi se pierd pachete'); plt.show()

## B.6 Problema 6 — proiectați (și rulați) un protocol
Mai jos este *„un răspuns posibil”* al seminarului, transformat în cod care rulează: un **serviciu de oră** peste UDP.

* **Format**: linii ASCII terminate cu CR LF. Cerere `TIME?`; răspuns `TIME 2026-10-06T09:30:00Z` (ISO 8601, UTC); altfel `ERR 400 unknown request`.
* **Ordine**: clientul vorbește primul; exact un răspuns pentru fiecare cerere.
* **Acțiuni**: clientul așteaptă 2 s, reîncearcă, renunță după 3 încercări.

Serverului i se poate cere să **piardă** primele câteva cereri, ca să vedeți cum funcționează regula de timeout și reîncercare.

In [ ]:
import datetime

class TimeServer(threading.Thread):
    def __init__(self, port=0, drop_first=0):
        super().__init__(daemon=True)
        self.sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        self.sock.bind(('127.0.0.1', port)); self.port = self.sock.getsockname()[1]
        self.drop_first, self.running = drop_first, True
    def run(self):
        while self.running:
            try:
                data, addr = self.sock.recvfrom(1500)
            except OSError:
                break
            if self.drop_first > 0:          # simulăm pierderea primelor cereri
                self.drop_first -= 1
                print(f'   [server] cererea {data!r} s-a PIERDUT în rețea'); continue
            line = data.decode('ascii', 'replace')
            if line == 'TIME?\r\n':
                now = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
                reply = f'TIME {now}\r\n'
            else:
                reply = 'ERR 400 unknown request\r\n'
            self.sock.sendto(reply.encode('ascii'), addr)
    def stop(self):
        self.running = False; self.sock.close()

def time_client(port, request='TIME?', timeout=2.0, tries=3):
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM); s.settimeout(timeout)
    try:
        for attempt in range(1, tries + 1):
            s.sendto((request + '\r\n').encode('ascii'), ('127.0.0.1', port))
            try:
                reply, _ = s.recvfrom(1500)
                print(f'   [client] încercarea {attempt}: {reply.decode().strip()}'); return reply
            except socket.timeout:
                print(f'   [client] încercarea {attempt}: niciun răspuns după {timeout} s')
        print('   [client] renunț → raportez eșecul utilizatorului'); return None
    finally:
        s.close()

srv = TimeServer(); srv.start()
print('Cerere corectă:'); time_client(srv.port)
print('Încălcarea formatului:'); time_client(srv.port, 'HELLO')
srv.stop()

srv = TimeServer(drop_first=2); srv.start()
print('Primele 2 cereri se pierd (timeout scurtat la 0,5 s pentru demonstrație):'); time_client(srv.port, timeout=0.5)
srv.stop()
srv = TimeServer(drop_first=5); srv.start()
print('Serverul este inaccesibil o vreme (5 pierderi):'); time_client(srv.port, timeout=0.5)
srv.stop()

### De ce contează idempotența — întrebarea „transferă 100 lei” din curs
A întreba ora de două ori nu strică nimic (cererea este **idempotentă**). Un transfer nu este. Mai jos, **răspunsul** se pierde, clientul reîncearcă, iar banii pleacă de două ori. Un **identificator de cerere** îi permite serverului să recunoască repetarea.

In [ ]:
class Bank:
    def __init__(self):
        self.balance, self.seen = 1000, {}
    def handle(self, msg, use_ids):
        req_id, cmd, amount = msg.split()
        if use_ids and req_id in self.seen:
            return self.seen[req_id]                         # cerere repetată: același răspuns, niciun transfer nou
        self.balance -= int(amount)
        reply = f'OK balance={self.balance}'
        self.seen[req_id] = reply
        return reply

def transfer(bank, use_ids, lose_first_reply=True):
    for attempt in (1, 2):
        reply = bank.handle('req-42 TRANSFER 100', use_ids)
        if lose_first_reply and attempt == 1:
            print(f'   încercarea 1: serverul a executat ({reply}), dar răspunsul s-a PIERDUT → clientul reîncearcă')
            continue
        print(f'   încercarea {attempt}: {reply}'); break

print('Fără identificatori de cerere:'); b1 = Bank(); transfer(b1, use_ids=False); print('   sold final', b1.balance, '← 200 de lei au plecat din cont!')
print('Cu identificatori de cerere:');    b2 = Bank(); transfer(b2, use_ids=True);  print('   sold final', b2.balance)

## B.7 Problema 7 — costul antetelor (TCP 20 B + IPv4 20 B + antet Ethernet 14 B + trailer 4 B)

In [ ]:
HDRS = {'TCP': 20, 'IPv4': 20, 'Ethernet header': 14, 'Ethernet trailer': 4}
for msg in (100, 1460):
    frame = msg + sum(HDRS.values())
    print(f'mesaj de {msg:5d} octeți → cadru de {frame} octeți, supraîncărcare {sum(HDRS.values())/frame:.1%}')
check('1. lungimea cadrului', 100 + 58, 158); check('2. lungimea cadrului', 1460 + 58, 1518)
print('3. comutatorul citește: Ethernet · routerul: Ethernet + IP · gazda destinație: toate antetele')

## B.8 Problema 8 — citirea unui traceroute
Ieșirea inventată din fișa de seminar este analizată automat: rețelele sunt recunoscute după numele de domeniu, iar cel mai mare salt de RTT este localizat și transformat într-o distanță.

In [ ]:
TRACE = """
 1  gw.campus.example.edu (10.20.0.1)          0.4 ms   0.3 ms   0.4 ms
 2  core1.campus.example.edu (10.0.0.1)        0.7 ms   0.6 ms   0.7 ms
 3  cta-1.nren.example.net (192.0.2.17)        1.9 ms   1.8 ms   2.1 ms
 4  buc-2.nren.example.net (192.0.2.41)        5.2 ms   5.1 ms   5.4 ms
 5  * * *
 6  vie-1.transit.example.net (198.51.100.9)  22.8 ms  23.1 ms  22.7 ms
 7  fra-3.transit.example.net (198.51.100.33) 34.6 ms  34.2 ms  61.0 ms
 8  nyc-1.transit.example.net (198.51.100.77) 118.3 ms 117.9 ms 118.6 ms
 9  edge.cdn.example.com (203.0.113.5)        117.5 ms 118.1 ms 117.8 ms
"""

def parse_traceroute(text):
    hops = []
    for line in text.strip().splitlines():
        m = re.match(r'\s*(\d+)\s+(.*)', line)
        if not m: continue
        hop, rest = int(m.group(1)), m.group(2)
        name_ip = re.search(r'(\S+)\s+\(([\d.]+)\)', rest)
        ip_only = re.search(r'^\s*([\d.]+)\s', rest)
        name = name_ip.group(1) if name_ip else (ip_only.group(1) if ip_only else '*')
        ip = name_ip.group(2) if name_ip else (ip_only.group(1) if ip_only else None)
        rtts = [float(x) for x in re.findall(r'([\d.]+)\s*ms', rest)]
        hops.append({'hop': hop, 'name': name, 'ip': ip, 'rtts': rtts})
    return hops

def network_of(name):
    parts = name.split('.')
    return '.'.join(parts[-3:-2]) if len(parts) >= 3 else name

hops = parse_traceroute(TRACE)
print(f'{"hop":>3s}  {"router":30s} {"rețea":10s} {"RTT median":>10s}')
for h in hops:
    med = np.median(h['rtts']) if h['rtts'] else float('nan')
    print(f'{h["hop"]:3d}  {h["name"]:30s} {network_of(h["name"]) if h["ip"] else "-":10s} {med:10.1f}')
nets = []
for h in hops:
    if h['ip'] and (not nets or nets[-1] != network_of(h['name'])):
        nets.append(network_of(h['name']))
print('\n1. Rețele traversate:', ' → '.join(nets), f'({len(nets)})')

answered = [h for h in hops if h['rtts']]
jumps = [(b['hop'], a['hop'], min(b['rtts']) - min(a['rtts'])) for a, b in zip(answered, answered[1:])]
to_h, from_h, jump = max(jumps, key=lambda j: j[2])
print(f'2. Cel mai mare salt: hop {from_h} → {to_h}, +{jump:.0f} ms RTT → într-un sens ≈ {jump/2:.0f} ms '
      f'→ cel mult {jump/2*200:.0f} km de fibră (Frankfurt → New York)')
print('3. „* * *”: acel router nu răspunde la probe; hop-urile de după răspund, deci calea nu este întreruptă.')
print('4. 61 ms la hop-ul 7: o coadă de moment sau un răspuns lent al routerului — un eșantion din trei este zgomot.')
print('5. Hop 9 < hop 8: posibil; pachete diferite în momente diferite, iar routerele răspund cu prioritate scăzută.')

plt.plot([h['hop'] for h in answered], [np.median(h['rtts']) for h in answered], 'o-')
for h in answered:
    plt.annotate(h['name'].split('.')[0], (h['hop'], np.median(h['rtts'])), fontsize=7,
                 xytext=(3, -10), textcoords='offset points')
plt.xlabel('hop'); plt.ylabel('RTT median (ms)'); plt.title('RTT de-a lungul căii'); plt.show()

**Temă (Seminarul 1):** (1) exercițiile de la curs fără răspunsuri — apoi comparați cu A.8; (2) Problema 4 cu activitate de 50 % și 15 utilizatori — vezi B.4; (3) comparați **RFC 867** (*Daytime Protocol*) cu designul din B.6: ce a lăsat autorul nespecificat?

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Tema 3 — RFC 867 vs. designul nostru:** 

---
# Partea C · Laboratorul 1 (Linux) — explorarea rețelei din linia de comandă
Tot ce urmează rulează pe mașina virtuală Colab. **Reguli de conduită:** sondați doar servere publice bine cunoscute, cu numerele mici de pachete date aici. Nu scanați intervale de adrese sau porturi.

## Sarcina 1 — Cine sunt eu?

In [ ]:
sh('hostname')
sh('ip -br link')
sh('ip -br addr')
sh('ip addr show')

In [ ]:
# Aceleași informații citite direct din nucleu (/sys/class/net)
IFACE, GATEWAY, MY_IP = default_route()
print(f'Interfața spre Internet: {IFACE}   IPv4-ul meu: {MY_IP}   gateway implicit: {GATEWAY}\n')
for ifc in sorted(os.listdir('/sys/class/net')):
    rd = lambda f: open(f'/sys/class/net/{ifc}/{f}').read().strip() if os.path.exists(f'/sys/class/net/{ifc}/{f}') else '?'
    wireless = os.path.exists(f'/sys/class/net/{ifc}/wireless')
    print(f'{ifc:10s} MAC {rd("address"):17s}  MTU {rd("mtu"):5s}  stare {rd("operstate"):8s}  {"fără fir" if wireless else "cu fir/virtuală"}')
if MY_IP:
    net = ipaddress.ip_interface(re.search(rf'inet ({re.escape(MY_IP)}/\d+)', sh('ip -4 addr', show=False)).group(1))
    print(f'\nRețeaua mea: {net.network}  → primii {net.network.prefixlen} biți sunt comuni tuturor gazdelor de pe legătura mea')

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q1 — interfețele; care vă conectează; cu fir sau fără fir?:** 
- **Q2 — IPv4/prefix, MAC, MTU; cărui strat îi aparține fiecare adresă?:** 
- **Q3 — ce parte este aceeași cu adresa unui vecin? (în Colab: comparați cu VM-ul Colab al unui coleg):** 

## Sarcina 2 — Drumul spre exterior

In [ ]:
sh('ip route')
sh('ip route get 1.1.1.1')
sh('ip neigh')
sh('cat /etc/resolv.conf')
if have('resolvectl'):
    sh('resolvectl status | head -20')
n_routes = len([l for l in sh('ip route', show=False).splitlines() if l.strip()])
print(f'Această gazdă are {n_routes} rute; un router din nucleul rețelei are ~1 000 000 (tabela IPv4 completă).')

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q4 — gateway-ul implicit; numărul de rute și de ce sunt suficiente atât de puține:** 
- **Q5 — serverul DNS; este în propria voastră rețea?:** 

## Sarcina 3 — Accesibilitate și timp dus-întors (RTT)
`ping` trimite cereri ICMP Echo. Dacă rețeaua Colab blochează ICMP, veți vedea pierderi de 100 % — **lipsa răspunsului este o dovadă slabă**. Notebook-ul măsoară atunci RTT-ul printr-un handshake TCP (durata lui `connect()`).

In [ ]:
UNI_SERVER = 'www.ubbcluj.ro'      # ← puneți aici serverul web al universității voastre
TARGETS = [GATEWAY or '1.1.1.1', UNI_SERVER, 'www.kernel.org']

def ping_stats(out):
    m = re.search(r'(\d+) packets transmitted, (\d+) received', out)
    r = re.search(r'= ([\d.]+)/([\d.]+)/([\d.]+)/([\d.]+) ms', out)
    ttl = re.search(r'ttl=(\d+)', out)
    return {'sent': int(m.group(1)) if m else 0, 'recv': int(m.group(2)) if m else 0,
            'min': float(r.group(1)) if r else None, 'avg': float(r.group(2)) if r else None,
            'max': float(r.group(3)) if r else None, 'ttl': int(ttl.group(1)) if ttl else None}

results = {}
for t in TARGETS:
    st = ping_stats(sh(f'ping -c 5 -W 2 {t}', timeout=30))
    if st['recv'] == 0 and t != GATEWAY:
        print(f'   ICMP nu a primit răspuns → măsurăm în schimb cu handshake-uri TCP')
        rt = tcp_rtt(t, 443, 5)
        if rt:
            st.update(min=min(rt), avg=sum(rt)/len(rt), max=max(rt), method='TCP')
    results[t] = st

In [ ]:
print(f'{"destinație":22s} {"min":>8s} {"med":>8s} {"max":>8s} {"pierd.":>6s} {"TTL":>4s}  metoda')
for t, st in results.items():
    loss = f'{1 - st["recv"]/st["sent"]:6.0%}' if st['sent'] else '     –'
    f = lambda v: f'{v:8.2f}' if v is not None else '       –'
    print(f'{t:22s} {f(st["min"])} {f(st["avg"])} {f(st["max"])} {loss} {str(st["ttl"] or "–"):>4s}  {st.get("method","ICMP")}')

print('\nQ7 — limita distanței din RTT-ul minim (lumina în fibră: 200 km pe ms, RTT înseamnă dus ȘI întors):')
for t, st in results.items():
    if st['min']:
        print(f'   {t:22s} ≤ {st["min"]/2*200:8.0f} km')

print('\nQ9 — routerele de pe drum, estimate din TTL-ul răspunsurilor:')
for t, st in results.items():
    if st['ttl']:
        start = min(x for x in (64, 128, 255) if x >= st['ttl'])
        print(f'   {t:22s} TTL {st["ttl"]} → a pornit probabil de la {start} → ~{start - st["ttl"]} routere')

In [ ]:
# Q8 — schimbă dimensiunea pachetului RTT-ul? (ICMP, dacă se poate)
sh('ping -c 5 -W 2 www.kernel.org', timeout=30)
sh('ping -c 5 -W 2 -s 1400 www.kernel.org', timeout=30)

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q6 — tabelul și componenta dominantă a întârzierii:** 
- **Q7 — limita superioară a distanței; de ce doar o limită superioară?:** 
- **Q8 — efectul lui -s 1400; ce întârziere depinde de dimensiune?:** 
- **Q9 — routerele dintre voi și fiecare destinație:** 

## Sarcina 4 — Calea: o rețea de rețele
`traceroute` trimite implicit probe UDP; `-I` trimite ICMP Echo (ca `tracert` din Windows); `-T` trimite TCP SYN la portul 443, proba cel mai des lăsată să treacă de firewall-urile din cloud. Liniile cu `*` sunt routere care nu răspund.

In [ ]:
DEST = 'www.kernel.org'
out_udp = sh(f'traceroute -n -q 1 -w 2 -m 20 {DEST}', timeout=90)
out_tcp = sh(f'{SUDO}traceroute -T -p 443 -n -q 3 -w 2 -m 20 {DEST}', timeout=120)

In [ ]:
if have('mtr'):
    sh(f'mtr -rwnc 5 {DEST}', timeout=120)

In [ ]:
# Analizăm cel mai bun traseu: nume, deținătorii blocurilor de adrese (RDAP), salturi de RTT
import urllib.request

def rdap_owner(ip):
    try:
        if ipaddress.ip_address(ip).is_private: return 'adresă privată (acest sit)'
        with urllib.request.urlopen(f'https://rdap.org/ip/{ip}', timeout=6) as r:
            j = json.load(r)
        return f"{j.get('name','?')} ({j.get('country','?')})"
    except Exception as e:
        return f'? ({type(e).__name__})'

def rdns(ip):
    try: return socket.gethostbyaddr(ip)[0]
    except Exception: return ''

trace = max([parse_traceroute(o) for o in (out_udp, out_tcp)], key=lambda h: sum(1 for x in h if x['ip']))
print(f'{"hop":>3s} {"adresa":15s} {"RTT ms":>7s}  {"nume":40s} deținătorul blocului de adrese')
for h in trace:
    if not h['ip']:
        print(f'{h["hop"]:3d} {"*":15s}'); continue
    print(f'{h["hop"]:3d} {h["ip"]:15s} {min(h["rtts"]) if h["rtts"] else float("nan"):7.2f}  '
          f'{rdns(h["ip"])[:40]:40s} {rdap_owner(h["ip"])}')
ans = [h for h in trace if h['rtts']]
if len(ans) > 1:
    plt.plot([h['hop'] for h in ans], [min(h['rtts']) for h in ans], 'o-')
    plt.xlabel('hop'); plt.ylabel('RTT minim (ms)'); plt.title(f'Calea spre {DEST}'); plt.show()

In [ ]:
# Q13 — o destinație din România vs. una de pe alt continent: unde se despart căile?
def hop_ips(dest):
    o = sh(f'{SUDO}traceroute -T -p 443 -n -q 1 -w 2 -m 15 {dest}', timeout=90, show=False)
    return [h['ip'] for h in parse_traceroute(o)]
ro, far = hop_ips('www.ubbcluj.ro'), hop_ips('www.usp.br')
common = 0
for a, z in zip(ro, far):
    if a != z: break
    common += 1
print('România  :', ro); print('Am. de Sud:', far)
print(f'Cele două căi au în comun primele {common} hop-uri (aceeași ieșire din acest centru de date).')

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q10 — numărul de hop-uri; concordanța cu Q9:** 
- **Q11 — rețelele traversate (desenați-le ca în Figura 1):** 
- **Q12 — cea mai mare creștere a RTT; ce pas geografic?:** 
- **Q13 — unde se despart cele două căi și de ce:** 

## Sarcina 5 — Nume și adrese (DNS)

In [ ]:
sh('dig www.kernel.org')
sh('dig +short www.kernel.org')
sh('dig +short AAAA www.kernel.org')
router = next((h['ip'] for h in trace if h['ip'] and not ipaddress.ip_address(h['ip']).is_private), '1.1.1.1')
sh(f'dig -x {router} +short')
for i in (1, 2):
    qt = re.search(r'Query time: (\d+) msec', sh('dig www.kernel.org', show=False))
    print(f'interogarea {i}: {qt.group(1) if qt else "?"} ms')

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q14 — adresele (A și AAAA); timpul interogării a doua oară și de ce:** 

## Sarcina 6 — Un protocol de mână
Mai întâi priviți cum vorbește `curl` HTTP (`>` trimis, `<` primit); apoi fiți voi browserul, cu `nc` și cu un socket Python brut; apoi stricați protocolul cu `HELLO`.

In [ ]:
sh('curl -sv http://example.org/ -o /dev/null 2>&1 | grep -E "^[<>*]" | head -30')
sh(r"printf 'GET / HTTP/1.1\r\nHost: example.org\r\nConnection: close\r\n\r\n' | nc -w 5 example.org 80 | head -12")

In [ ]:
def raw_http(host, request_lines, port=80):
    """Deschide o conexiune TCP și trimite exact liniile date, fiecare terminată cu CR LF."""
    data = b''
    try:
        with socket.create_connection((host, port), timeout=8) as s:
            s.sendall(('\r\n'.join(request_lines) + '\r\n').encode())
            try:
                while chunk := s.recv(4096):
                    data += chunk
            except socket.timeout:
                pass
    except OSError as e:
        print(f'   conectarea la {host}:{port} a eșuat: {e}')
    return data.decode('latin-1')

good = raw_http('example.org', ['GET / HTTP/1.1', 'Host: example.org', 'Connection: close', ''])
bad = raw_http('example.org', ['HELLO', ''])
print('Cerere corectă →', good.splitlines()[0] if good else '(niciun răspuns)')
print('HELLO          →', bad.splitlines()[0] if bad else '(conexiune închisă fără răspuns)')

In [ ]:
# ss -tn cât timp o conexiune este deschisă
try:
    s = socket.create_connection(('example.org', 80), timeout=8)
    sh('ss -tn')
    s.close()
except OSError as e:
    print('nu m-am putut conecta:', e); sh('ss -tn')

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q15 — prima linie a ambelor răspunsuri; formatul, ordinea și acțiunile din schimbul vostru; ce a făcut serverul cu HELLO?:** 

## Sarcina 7 — Debitul
`TEST_URL` este un fișier de test; înlocuiți-l cu cel dat de cadrul didactic. Repetăm măsurătoarea, apoi rulăm patru descărcări simultan, ca și cum patru colegi ar descărca în același timp.

In [ ]:
TEST_URL = 'https://speed.cloudflare.com/__down?bytes=25000000'   # fișier de test de 25 MB

def curl_speed(url=TEST_URL, show=True):
    out = sh(f"curl -s -o /dev/null -w '%{{speed_download}} %{{time_total}}' '{url}'", timeout=120, show=False)
    try:
        bps, t = map(float, out.split()[:2])
    except ValueError:
        bps = 0
    if bps == 0:
        print('descărcarea a eșuat — verificați TEST_URL'); return None
    if show: print(f'viteză {bps:,.0f} B/s = {bps*8/1e6:7.1f} Mbit/s   timp {t:.2f} s')
    return bps * 8 / 1e6

single = [curl_speed() for _ in range(3)]

In [ ]:
res = []
ths = [threading.Thread(target=lambda: res.append(curl_speed(show=False))) for _ in range(4)]
[t.start() for t in ths]; [t.join() for t in ths]
res = [r for r in res if r]
single = [s for s in single if s]
if res and single:
    print('singur         :', ', '.join(f'{x:.0f}' for x in single), 'Mbit/s')
    print('4 în paralel   :', ', '.join(f'{x:.0f}' for x in res), f'Mbit/s  (suma {sum(res):.0f})')
    print('Dacă suma rămâne aproape de valoarea individuală, gâtuirea comună este aproape de voi (legătura VM-ului);')
    print('dacă fiecare descărcare paralelă e la fel de rapidă ca una singură, gâtuirea era fluxul unic sau serverul.')

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q16 — rata în Mbit/s; unde este gâtuirea și ce dovezi aveți?:** 

## Sarcina 8 — Vizualizarea pachetelor (opțional)
`tcpdump -e` afișează și antetul de nivel legătură (MAC): încapsularea devine vizibilă. Capturăm în timp ce generăm puțin trafic.

In [ ]:
def capture_while(action, bpf, count=8, iface=None, seconds=6):
    iface = iface or IFACE
    p = subprocess.Popen(f'{SUDO}timeout {seconds} tcpdump -i {iface} -n -e -c {count} {bpf}',
                         shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(1.0)
    try:
        action()
    except Exception as e:
        print('acțiunea a eșuat:', e)
    print(p.communicate()[0])

capture_while(lambda: sh(f'ping -c 3 -W 1 {GATEWAY or "1.1.1.1"}', show=False), 'icmp')
capture_while(lambda: raw_http('example.org', ['GET / HTTP/1.1', 'Host: example.org', 'Connection: close', '']),
              "'tcp port 80'")

---
# Partea D · Laboratorul 1 (Windows) — aceleași idei, în stil Windows
Colab rulează Linux, așa că cmdlet-urile PowerShell din fișa Windows (`Get-NetIPConfiguration`, `Test-NetConnection`, …) nu sunt disponibile aici. Această parte oferă (1) **tabelul comparativ** sub formă de autotest, (2) echivalente Python ale sarcinilor Windows care nu au un geamăn direct în Linux și (3), opțional, **PowerShell 7 real** pe VM, pentru a rula scriptul `TcpClient` din fișă exact așa cum este scris.

## D.1 Linux ↔ Windows, din memorie
Completați dicționarul, apoi rulați celula ca să vă verificați (Sarcina 8 din fișa Windows).

In [ ]:
TABLE = {  # scop: (Linux, Windows clasic, Windows PowerShell)
    'Interfețe și adrese': ('ip addr', 'ipconfig /all', 'Get-NetIPConfiguration'),
    'Tabela de retransmitere':        ('ip route', 'route print', 'Get-NetRoute'),
    'Ruta spre o destinație': ('ip route get', '–', 'Find-NetRoute'),
    'Vecinii de pe legătură':   ('ip neigh', 'arp -a', 'Get-NetNeighbor'),
    'Accesibilitate, RTT':        ('ping -c', 'ping -n', 'Test-Connection'),
    'Calea':                    ('traceroute, mtr', 'tracert, pathping', 'Test-NetConnection -TraceRoute'),
    'Rezolvarea numelor':              ('dig', 'nslookup', 'Resolve-DnsName'),
    'Testarea unui port TCP':          ('nc -vz host port', '–', 'Test-NetConnection -Port'),
    'Conexiuni deschise':         ('ss -tn', 'netstat -ano', 'Get-NetTCPConnection'),
    'Client HTTP':              ('curl -v', 'curl.exe -v', 'Invoke-WebRequest'),
    'Captură de pachete':           ('tcpdump, Wireshark', 'pktmon (admin), Wireshark', '–'),
}
MY_ANSWERS = {   # ← scrieți răspunsurile voastre pentru coloana Windows clasic
    'Tabela de retransmitere': 'route print',
    'Rezolvarea numelor': '',
    'Conexiuni deschise': '',
    'Vecinii de pe legătură': '',
}
for k, mine in MY_ANSWERS.items():
    right = TABLE[k][1]
    print(f"{'✅' if mine.strip().lower() == right.lower() else '❌'} {k:26s} voi: {mine or '(gol)':15s} răspuns: {right}")

## D.2 `ipconfig /all` — aceleași informații adunate într-un loc
Windows scrie prefixul ca **mască de subrețea**; `255.255.255.0` este `/24`.

In [ ]:
def ipconfig_like():
    out = sh('ip -4 -o addr show', show=False)
    dns = re.findall(r'^nameserver\s+(\S+)', open('/etc/resolv.conf').read(), re.M)
    for m in re.finditer(r'\d+: (\S+)\s+inet (\S+)', out):
        ifc, cidr = m.group(1), ipaddress.ip_interface(m.group(2))
        mac = open(f'/sys/class/net/{ifc}/address').read().strip().replace(':', '-').upper()
        print(f'Adapter {ifc}:')
        print(f'   Physical Address. . . : {mac}')
        print(f'   IPv4 Address. . . . . : {cidr.ip}')
        print(f'   Subnet Mask . . . . . : {cidr.netmask}   (= /{cidr.network.prefixlen})')
        print(f'   Default Gateway . . . : {GATEWAY if ifc == IFACE else ""}')
        print(f'   DNS Servers . . . . . : {", ".join(dns) if ifc == IFACE else ""}\n')
ipconfig_like()
for mask in ['255.255.255.0', '255.255.0.0', '255.255.255.192']:
    print(f'{mask:16s} = /{ipaddress.ip_network("0.0.0.0/" + mask).prefixlen}')

## D.3 `Test-NetConnection -Port` și `Resolve-DnsName`
Portul 80 răspunde imediat. Portul 81 de pe example.org este ignorat în tăcere, așa că clientul își tot retransmite SYN-ul: Windows renunță după aproximativ **21 s** (3 + 6 + 12 s), Linux după aproximativ **127 s** (6 reîncercări). Aici limităm așteptarea la 10 s.

In [ ]:
def test_net_connection(host, port, timeout=10):
    ip = resolve(host)
    if not ip: return
    t0 = time.perf_counter()
    try:
        socket.create_connection((ip, port), timeout=timeout).close(); ok = True
    except Exception as e:
        ok, err = False, type(e).__name__
    dt = time.perf_counter() - t0
    print(f'ComputerName {host}  RemoteAddress {ip}  RemotePort {port}  '
          f'TcpTestSucceeded {ok}  ({dt:.2f} s{"" if ok else ", " + err})')

test_net_connection('example.org', 80)
test_net_connection('example.org', 81)
print('Reîncercări SYN în Linux pe acest VM:', open('/proc/sys/net/ipv4/tcp_syn_retries').read().strip())

In [ ]:
def resolve_dnsname(name, family=socket.AF_UNSPEC):
    seen = set()
    try:
        infos = socket.getaddrinfo(name, None, family)
    except OSError as e:
        print(f'{name}: {e}'); return
    for fam, *_, addr in infos:
        if addr[0] not in seen:
            seen.add(addr[0]); print(f'{name:20s} {"AAAA" if fam == socket.AF_INET6 else "A":4s} {addr[0]}')
resolve_dnsname('www.kernel.org')
sh('dig www.kernel.org | grep -A4 "ANSWER SECTION"')   # coloana TTL spune cât timp poate fi păstrat răspunsul în cache

## D.4 HTTP de mână — scriptul `TcpClient` din fișa Windows
Întâi versiunea Python (aceiași pași: deschide conexiunea TCP → scrie linii terminate cu CR LF → citește până la capăt).

In [ ]:
for first_line in ('GET / HTTP/1.1', 'HELLO'):
    reply = raw_http('example.org', [first_line, 'Host: example.org', 'Connection: close', ''])
    print(f'{first_line:15s} → {reply.splitlines()[0] if reply else "(niciun răspuns)"}')

**Opțional: rulați exact scriptul PowerShell.** Setați `INSTALL_PWSH = True` (durează cam un minut). PowerShell 7 rulează pe Linux și include `System.Net.Sockets.TcpClient` din .NET, așa că scriptul din fișa Windows rulează neschimbat. (Cmdlet-urile exclusiv Windows, precum `Test-NetConnection`, tot nu sunt disponibile.)

In [ ]:
INSTALL_PWSH = False

if INSTALL_PWSH and not have('pwsh'):
    ver = re.search(r'VERSION_ID="([\d.]+)"', open('/etc/os-release').read()).group(1)
    sh(f'wget -q https://packages.microsoft.com/config/ubuntu/{ver}/packages-microsoft-prod.deb -O /tmp/ms.deb && '
       f'{SUDO}dpkg -i /tmp/ms.deb > /dev/null && {SUDO}apt-get -qq update > /dev/null && '
       f'{SUDO}apt-get -qq install -y powershell > /dev/null; pwsh -Version', timeout=600)

PS_SCRIPT = r"""
$client = New-Object System.Net.Sockets.TcpClient('example.org', 80)
$stream = $client.GetStream()
$writer = New-Object System.IO.StreamWriter($stream)
$writer.NewLine = "`r`n"; $writer.AutoFlush = $true
$reader = New-Object System.IO.StreamReader($stream)
$writer.WriteLine('GET / HTTP/1.1')
$writer.WriteLine('Host: example.org')
$writer.WriteLine('Connection: close')
$writer.WriteLine('')
$reader.ReadToEnd().Split("`n")[0..8] -join "`n"
$client.Close()
"""
if have('pwsh'):
    open('/tmp/http.ps1', 'w').write(PS_SCRIPT)
    sh('pwsh -NoProfile -File /tmp/http.ps1', timeout=60)
else:
    print('PowerShell nu este instalat (setați INSTALL_PWSH = True pentru a-l încerca).')

## D.5 Întrebări specifice fișei Windows
* **Q3 (Windows)** — A fost adresa configurată prin DHCP? În Linux: `ip addr` arată `dynamic` și durata de viață rămasă (`valid_lft`).
* **Q9 (Windows)** — Windows Firewall nu răspunde implicit la ping de la alte mașini. Ce dovedește un ping eșuat?
* **Q10 (Windows)** — TTL-uri inițiale: Linux și echipamentele de rețea 64 sau 255, Windows 128.
* **Q13 (Windows)** — `pathping` arată pierderi la un hop intermediar, dar nu la destinație: există o problemă reală?

In [ ]:
sh(f'ip addr show {IFACE} | grep -E "inet |valid_lft"')
print('TTL-ul inițial pe acest VM Linux:', open('/proc/sys/net/ipv4/ip_default_ttl').read().strip())

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q3 (Windows) — DHCP?:** 
- **Q9 (Windows) — ce dovedește un ping eșuat:** 
- **Q13 (Windows) — pierderi doar la un hop intermediar:** 

---
### Săptămâna viitoare
Cursul 2: arhitecturi stratificate în detaliu, OSI vs. TCP/IP și o tratare cantitativă a întârzierii, pierderilor și debitului. Laboratorul 2: captura de pachete — vezi notebook-ul din **Săptămâna 2**.